# Auditory-Evoked Pupillary Responses (AEPR) - Kaggle Training Runner

Automated execution environment for reproducible physiological deep learning training on Kaggle GPU.

In [ ]:
# ====================================================================
# Cell 1: Environment, Pinned Dependency Checks & GPU Verification
# ====================================================================
import sys
import os
import json
import torch
import numpy as np
import pandas as pd
import scipy
import sklearn

print('=' * 60)
print('ENVIRONMENT & HARDWARE DIAGNOSTICS')
print('=' * 60)
print(f'Python Version:       {sys.version.split()[0]}')
print(f'PyTorch Version:      {torch.__version__}')
print(f'NumPy Version:        {np.__version__}')
print(f'Pandas Version:       {pd.__version__}')
print(f'SciPy Version:        {scipy.__version__}')
print(f'Scikit-Learn Version: {sklearn.__version__}')

cuda_available = torch.cuda.is_available()
print(f'\nCUDA Available:       {cuda_available}')
if cuda_available:
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device Name:      {gpu_name}')
    print(f'GPU Device Count:     {gpu_count}')
    print(f'GPU Total Memory:     {gpu_mem:.2f} GB')
    print(f'CUDA Version:         {torch.version.cuda}')
else:
    print('WARNING: Running in CPU mode. GPU was not detected!')
print('=' * 60)


In [ ]:
# ====================================================================
# Cell 2: Verify Mounted Kaggle Dataset
# ====================================================================
from pathlib import Path

input_dir = Path('/kaggle/input')
print(f'Mounted datasets in {input_dir}:')
for p in input_dir.glob('*'):
    print(f'  - {p.name} ({len(list(p.glob("**/*")))} files)')

aepr_dataset_dir = Path('/kaggle/input/aepr-pupillometry-dataset')
if not aepr_dataset_dir.exists():
    # Fallback search if dataset slug differs slightly
    candidates = list(input_dir.glob('*pupil*')) + list(input_dir.glob('*aepr*'))
    if candidates:
        aepr_dataset_dir = candidates[0]
        print(f'Using detected dataset path: {aepr_dataset_dir}')
    else:
        print(f'Notice: {aepr_dataset_dir} not directly found, scanning /kaggle/input')
        aepr_dataset_dir = input_dir


In [ ]:
# ====================================================================
# Cell 3: Unpack Local Codebase (src/)
# ====================================================================
import base64
import tarfile
import io

src_payload = '''...'''

tar_bytes = base64.b64decode(src_payload.encode('ascii'))
with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r:gz') as tar:
    tar.extractall(path='/kaggle/working')

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

import src
print(f'Successfully loaded src package (version {src.__version__}) into Kaggle environment.')


In [ ]:
# ====================================================================
# Cell 4: Execute Universal Training / Smoke Test Runner
# ====================================================================
from src.train import run_smoke_test

data_path = aepr_dataset_dir
working_dir = Path('/kaggle/working')

metrics = run_smoke_test(data_dir=data_path, output_dir=working_dir, seed=42)
print('Execution returned metrics:', metrics)


In [ ]:
# ====================================================================
# Cell 5: Validate Saved Outputs and Reproducibility Logs
# ====================================================================
print('\nGenerated Outputs in /kaggle/working:')
for f in sorted(list(working_dir.glob('*'))):
    if f.is_file():
        print(f'  - {f.name} ({f.stat().st_size} bytes)')

meta_file = working_dir / 'run_metadata.json'
if meta_file.exists():
    print('\n--- RUN METADATA ---')
    with open(meta_file) as f:
        print(f.read())

metric_file = working_dir / 'metrics.json'
if metric_file.exists():
    print('\n--- METRICS ---')
    with open(metric_file) as f:
        print(f.read())
